In [1]:
import os, sys
import pandas as pd
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
sys.path.append(os.path.abspath(".."))

from src.bm25 import load_bm25
from src.semantic import semantic_retriever_rag, load_semantic
from src.rag_pipeline import (
    build_context, build_prompt, semantic_rag_pipeline, hybrid_rag_pipeline,
    SYSTEM_PROMPT_V1, SYSTEM_PROMPT_V2, SYSTEM_PROMPT_V3,
)

/Users/lalosanchezmartinez/miniforge3/envs/dsci575_project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [3]:


llm_endpoint = HuggingFaceEndpoint(
    repo_id = "meta-llama/Meta-Llama-3-8B-Instruct",
    task = "text-generation",
    max_new_tokens = 200,
    huggingfacehub_api_token = hf_token,
)

llm = ChatHuggingFace(llm=llm_endpoint)

In [4]:
merged_df = pd.read_parquet("../data/processed/merged.parquet")
semantic_model, semantic_index, semantic_meta = load_semantic()
bm25 = load_bm25()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6774.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
test_docs = semantic_retriever_rag(
    query="good books for beginners learning investing",
    model=semantic_model,
    index=semantic_index,
    df=merged_df,
    top_k=3,
)

test_docs.head()

,parent_asin,product_title,review_title,review_text,rating,semantic_score
3496,B01AR5417A,Investment: A History (Columbia Business Schoo...,Long and tiresome,The book is long and tiresome. I didn't learn ...,3.0,0.491082
3021,B008Y1L7EA,Little Kids Big Money: Tools for Teaching Kid ...,Must Read For Parents,This book really makes a lot of sense on how t...,5.0,0.444927
3609,B005T75AF4,The Node Beginner Book,A good resource for starting Node,"A good resource for starting Node, but it fall...",3.0,0.438229


In [6]:

test_context = build_context(test_docs)
print(test_context)

Product ASIN: B01AR5417A
Title: Investment: A History (Columbia Business School Publishing)
Review Title: Long and tiresome
Rating: 3.0
Review Text: The book is long and tiresome. I didn't learn any interesting theory or facts. The main thesis is that nowadays more people invest while in old ages only the elite invested. But this something that I knew (and I believe everybody else knew) before reading this book.

---

Product ASIN: B008Y1L7EA
Title: Little Kids Big Money: Tools for Teaching Kid Friendly Finance
Review Title: Must Read For Parents
Rating: 5.0
Review Text: This book really makes a lot of sense on how to teach a child about money, how to budget money, and how to really think about money. I love how they talk about different incentives for different types of children. Yes, this author realizes that not all children are the same and what works for one, doesn't work for another.

---

Product ASIN: B005T75AF4
Title: The Node Beginner Book
Review Title: A good resource for st

## Prompts

We designed a prompt template to guide the model to answer the questions using only the retrieved reviews and metadata.

We tested three different prompt variants (V1, V2, V3) using the same context to isolate the effect of prompt wording. V1 produced more generic answers, while V3 was somewhat restrictive. Finally, V2 provided the best balance, generating clear, relevant, and context-grounded responses.

Therefore, V2 was selected as the default prompt in our RAG pipeline.

In [7]:
prompt_1 = build_prompt(
    query = "good books for beginners learning investing",
    context = test_context,
    system_prompt= SYSTEM_PROMPT_V1)

prompt_2 = build_prompt(
    query = "good books for beginners learning investing",
    context =test_context,
    system_prompt=SYSTEM_PROMPT_V2)

prompt_3 = build_prompt(
    query = "good books for beginners learning investing",
    context = test_context,
    system_prompt = SYSTEM_PROMPT_V3)

response_1 = llm.invoke(prompt_1)
response_2 = llm.invoke(prompt_2)
response_3 = llm.invoke(prompt_3)

print("V1:\n", response_1.content)

V1:
 Based on the reviews provided, I would not recommend the book "Investment: A History" (B01AR5417A) for beginners learning investing due to its lack of interesting theory or facts and its lengthy and tiresome content.

However, I would recommend considering "Little Kids Big Money" (B008Y1L7EA) as an indirect resource for learning investing concepts. While it focuses on teaching children about finance, it may provide some foundational concepts that can be applied to investing. However, this book is primarily geared towards parents and educators, and its primary focus is on teaching children financial literacy.

For programming knowledge on top of learning investing concepts, "The Node Beginner Book" (B005T75AF4) might be a good choice as a beginner's resource for starting Node. However it doesn't particularly help when it comes to a comprehensive learning of investing.


In [8]:
print("\nV2:\n", response_2.content)


V2:
 Unfortunately, the provided context does not contain enough relevant information about good books for beginners learning investing. 

However, it does mention Investment: A History (Columbia Business School Publishing) ASIN: B01AR5417A, but the review does not provide any information about it being suitable for beginners. The review discusses the book's content but seems to be disappointed by it, saying it didn't provide any interesting theory or facts.

If you're looking for a book on investing, it might be helpful to explore other reviews and products.


In [9]:
print("\nV3:\n", response_3.content)


V3:
 Based on the provided context, I would recommend the following books for beginners learning about investing:

1. Unfortunately, the first book, "Investment: A History (Columbia Business School Publishing)" (B01AR5417A), is not a suitable choice for beginners, as the reviewer found it long and tiresome without providing any interesting theory or facts.

2. Instead, consider "Little Kids Big Money: Tools for Teaching Kid Friendly Finance" (B008Y1L7EA), but this book is more focused on teaching children about finance rather than investing. However, it does cover some essential concepts related to money management and budgeting, which can be beneficial for beginners.

Since the provided context doesn't specifically mention books about investing for beginners, I must admit that I lack sufficient information to recommend a suitable book. If you provide more context or details, I'd be happy to give a more accurate answer.


## Pipeline description

We implemented two RAG flows in `src/rag_pipeline.py`, both using the same steps after retrieval: **context construction → prompt → LLM**.

1. **Semantic RAG** (`semantic_rag_pipeline`): top‑k from FAISS + embeddings only.
2. **Hybrid RAG** (`hybrid_rag_pipeline`): `hybrid_retriever` in `src/hybrid.py` fuses BM25 and semantic rankings (RRF), then the same context and prompt template as semantic RAG.

This lets us compare retrieval strategies while keeping the generator and prompts consistent.

In [ ]:
# Run full semantic RAG pipeline using selected prompt (V2)
result = semantic_rag_pipeline(
    query= "good books for beginners learning investing",
    llm = llm,
    model= semantic_model,
    index= semantic_index,
    df = merged_df,
    top_k = 3
    
)

print(result["answer"])

Unfortunately, there isn't enough context to directly answer this question. However, based on the provided information, I can say that one of the books, "Investment: A History" (B01AR5417A), does cover investing, but the reviewer felt it was a long and uninteresting read. This doesn't necessarily mean it's a bad resource for beginners, but the context doesn't provide enough information to confidently recommend it.


### Hybrid RAG (`hybrid_rag_pipeline`)

Same query and LLM as above; retrieval uses BM25 + semantic with RRF (`hybrid_retriever`). Compare `result_hybrid["answer"]` and `result_hybrid["docs"]` to the semantic-only run.

In [ ]:
result_hybrid = hybrid_rag_pipeline(
    query="good books for beginners learning investing",
    llm=llm,
    bm25=bm25,
    model=semantic_model,
    index=semantic_index,
    df=merged_df,
    top_k=3,
)
print(result_hybrid["answer"])

In [11]:
result["docs"][["parent_asin", "product_title"]]

,parent_asin,product_title
3496,B01AR5417A,Investment: A History (Columbia Business Schoo...
3021,B008Y1L7EA,Little Kids Big Money: Tools for Teaching Kid ...
3609,B005T75AF4,The Node Beginner Book


### Results

The pipeline successfully generated answers based on the provided reviews and metadata. The responses is generated based on the provided context and included relevant product suggestions.

After inspecting the retrieved documents, we were able to confirm that the model’s answers were supported by the input data provided. This demonstrates that the RAG pipeline is functioning correctly and producing recommendations based on the context.

### Limitations

One limitation of Rag is that it depends heavily on the quality and relevance of retrieved documents. If we do nto retrive useful results, the model may generate less relevant answers.

Also, some queries may not covere well all the available reviews,  which limits the system’s ability to provide strong recommendations.

Some improvements could include better retrieval tuning, filtering irrelevant documents, or combining semantic and keyword-based retrieval methods.